In [ ]:
import random
import numpy as np

def random_class():
    return random.randint(0, 1)


m = 4
n = 50

listX = []
for i in range(n):
    temp = []
    for j in range(m):
        temp.append(random_class())
    listX.append(temp)

X = np.asarray(listX)

listY = []
for i in range(n):
    listY.append(random_class())

Y = np.asarray(listY)

In [ ]:
print(X)
print(Y)

[[0 1 0 0]
 [1 1 0 1]
 [1 0 1 0]
 [1 0 0 1]
 [0 0 1 0]
 [1 0 1 1]
 [0 1 1 0]
 [1 0 1 1]
 [1 1 0 0]
 [0 1 0 1]
 [1 0 0 0]
 [1 1 1 1]
 [0 0 1 1]
 [1 1 0 1]
 [1 1 0 0]
 [0 0 0 0]
 [0 1 1 0]
 [0 0 0 1]
 [1 0 1 1]
 [1 0 0 1]
 [0 0 1 1]
 [1 0 0 1]
 [1 1 0 0]
 [1 1 0 1]
 [0 1 0 1]
 [1 1 0 0]
 [1 0 0 0]
 [0 1 1 1]
 [0 0 1 0]
 [1 1 0 1]
 [1 1 0 0]
 [1 0 0 0]
 [0 1 1 0]
 [1 1 0 0]
 [0 0 1 1]
 [0 0 1 0]
 [0 0 0 1]
 [1 0 0 0]
 [0 0 1 0]
 [1 1 0 0]
 [1 0 0 1]
 [1 1 1 0]
 [1 0 0 0]
 [1 1 1 1]
 [0 0 0 1]
 [1 0 1 0]
 [1 0 1 0]
 [1 1 1 1]
 [1 1 0 0]
 [0 0 0 0]]
[0 1 1 1 1 0 1 1 1 0 1 1 1 1 0 0 1 0 0 0 0 0 0 0 1 0 1 0 0 0 1 0 0 0 0 1 0
 1 1 0 1 0 1 1 1 1 0 1 0 1]


In [ ]:
import math

def S(x, y):
    if (x == 0 or y == 0): return 0
    return - x / (x + y) * math.log2(x / (x + y)) - y / (x + y) * math.log2(y / (x + y))

def entropy_calc(y):
    count0 = 0
    count1 = 0
    for i in range(len(y)):
        if y[i] == 0:
            count0 += 1
        else:
            count1 += 1
    return S(count0, count1)

def weighted_sum(col, X, Y):
    mask0 = X[:, col] == 0
    mask1 = X[:, col] == 1
    class0 = X[mask0, :]
    class1 = X[mask1, :]
    count0 = len(class0)
    count1 = len(class1)

    return (
        count0 / (count0 + count1) * entropy_calc(Y[mask0]) if count0 else 0
        +
        count1 / (count0 + count1) * entropy_calc(Y[mask1]) if count1 else 0
    )

def perform_split(col, X, Y):
    mask0 = X[:, col] == 0
    mask1 = X[:, col] == 1
    classX0 = X[mask0, :]
    classX1 = X[mask1, :]
    classY0 = Y[mask0]
    classY1 = Y[mask1]
    return classX0, classY0, classX1, classY1

def find_split(cols, X, Y):
    weighted_sums = [weighted_sum(col, X, Y) for col in cols]
    index = -1
    min_sum = 1e18
    for idx, sum in zip(range(len(weighted_sums)), weighted_sums):
        if (sum < min_sum):
            index = idx
            min_sum = sum

    return index, min_sum


In [ ]:
class Node:
    def __init__(self, idx=None, value=None, left=None, right=None, is_leaf=False, prediction=None):
        self.idx = idx
        self.left = left
        self.right = right
        self.value = value
        self.is_leaf = is_leaf
        self.prediction = prediction

In [ ]:
def majority_class(Y):
    if len(Y) == 0:
        return 0
    count0 = np.sum(Y == 0)
    count1 = np.sum(Y == 1)
    return 0 if count0 >= count1 else 1

def build_tree(X, Y, features_remaining, max_depth=5, current_depth=0):
    if len(np.unique(Y)) == 1:
        return Node(is_leaf=True, prediction=Y[0])

    if len(features_remaining) == 0 or current_depth >= max_depth:
        return Node(is_leaf=True, prediction=majority_class(Y))

    best_feature_index, min_weighted_sum = find_split(features_remaining, X, Y)

    if min_weighted_sum >= entropy_calc(Y) or best_feature_index == -1:
        return Node(is_leaf=True, prediction=majority_class(Y))

    classX0, classY0, classX1, classY1 = perform_split(best_feature_index, X, Y)

    new_features_remaining = [f for f in features_remaining if f != best_feature_index]

    left_child = build_tree(classX0, classY0, new_features_remaining, max_depth, current_depth + 1)
    right_child = build_tree(classX1, classY1, new_features_remaining, max_depth, current_depth + 1)

    return Node(idx=best_feature_index, left=left_child, right=right_child)

def predict_single(tree, sample):
    if tree.is_leaf:
        return tree.prediction

    if sample[tree.idx] == 0:
        return predict_single(tree.left, sample)
    else:
        return predict_single(tree.right, sample)

def predict(tree, X_test):
    predictions = []
    for sample in X_test:
        predictions.append(predict_single(tree, sample))
    return np.array(predictions)


In [ ]:
num_features = X.shape[1]

available_features = list(range(num_features))

decision_tree = build_tree(X, Y, available_features)

predictions = predict(decision_tree, X)

print("First 10 actual Y values:", Y[:10])
print("First 10 predicted values:", predictions[:10])

accuracy = np.mean(predictions == Y)
print(f"Training Accuracy: {accuracy*100:.2f}%")

First 10 actual Y values: [0 1 1 1 1 0 1 1 1 0]
First 10 predicted values: [0 1 1 1 0 1 0 1 1 0]
Training Accuracy: 52.00%
